# Notebook 5: Master Experiment Comparison

**Goal:** Synthesize findings across all 10 experiments.  
**Analyses:**
1. Summary table with all metrics
2. Variable importance — which single change drives the most improvement?
3. Radar charts per experiment
4. Latency vs. quality tradeoff
5. The headline finding

**Output:** `experiment_summary.json`, variable importance ranking

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from analysis.utils import (
    load_results, results_to_summary_df, METRIC_COLS,
    composite_score, radar_chart, apply_plot_style, save_output
)
apply_plot_style()

In [ ]:
results = load_results()
df = results_to_summary_df(results)
df['composite'] = composite_score(df)
df = df.sort_values('composite', ascending=False).reset_index(drop=True)

print(f'Loaded {len(df)} experiments')
df[['experiment_id', 'correctness', 'faithfulness', 'context_precision', 'context_recall', 'composite']].round(3)

## 1. Metric Heatmap — All Experiments

In [ ]:
metric_matrix = df.set_index('experiment_id')[METRIC_COLS]

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    metric_matrix.T, annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=0, vmax=1, linewidths=0.5, ax=ax
)
ax.set_title('Metric Scores Across All 10 Experiments')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 2. Variable Importance — What Changes Drive Improvement?

In [ ]:
# Each experiment changes one variable from a neighbor.
# Measure the delta in composite score for each change.
ablations = [
    ('LLM upgrade (mini → 4o)',   '01_baseline_naive', '02_better_llm'),
    ('Retrieval upgrade (dense → hybrid)', '02_better_llm', '03_hybrid_retrieval'),
    ('Add reranking',             '03_hybrid_retrieval', '04_hybrid_rerank'),
    ('Citation generation',       '04_hybrid_rerank', '05_citation_grounded'),
    ('Provider swap (GPT → Claude)', '04_hybrid_rerank', '06_cross_provider'),
    ('Query rewriting',           '04_hybrid_rerank', '07_query_rewrite'),
    ('HyDE transform',            '04_hybrid_rerank', '08_hyde'),
    ('Parent-child chunking',     '04_hybrid_rerank', '09_parent_child'),
    ('CRAG self-correction',      '04_hybrid_rerank', '10_crag'),
]

scores = df.set_index('experiment_id')['composite']
importance = []
for label, baseline_id, improved_id in ablations:
    if baseline_id in scores.index and improved_id in scores.index:
        delta = scores[improved_id] - scores[baseline_id]
        importance.append({'change': label, 'delta_composite': round(delta, 4)})

importance_df = pd.DataFrame(importance).sort_values('delta_composite', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if d >= 0 else 'red' for d in importance_df['delta_composite']]
ax.barh(importance_df['change'], importance_df['delta_composite'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Δ Composite Score')
ax.set_title('Variable Importance: Change in Composite Score')
plt.tight_layout()
plt.show()

print('Ranked by impact:')
print(importance_df.sort_values('delta_composite', ascending=False).to_string(index=False))

## 3. Radar Charts — Best vs Baseline

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'polar': True})

for ax, exp_id in zip(axes, [df.iloc[0]['experiment_id'], '01_baseline_naive']):
    row = df[df['experiment_id'] == exp_id].iloc[0]
    values = [row[m] for m in METRIC_COLS]
    radar_chart(ax, values, ['Correctness', 'Faithfulness', 'Precision', 'Recall'], title=exp_id)

plt.suptitle('Best Experiment vs. Baseline', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 4. Latency vs Quality Tradeoff

In [ ]:
fig = px.scatter(
    df,
    x='mean_latency_ms', y='composite',
    size='total_cost_usd', color='experiment_id',
    text='experiment_id',
    title='Latency vs Quality (bubble size = cost)',
    labels={'mean_latency_ms': 'Mean Latency (ms)', 'composite': 'Composite Quality'},
    width=900, height=550
)
fig.update_traces(textposition='top center')
fig.show()

## 5. Headline Finding

In [ ]:
best = df.iloc[0]
baseline = df[df['experiment_id'] == '01_baseline_naive'].iloc[0]
total_gain = best['composite'] - baseline['composite']

# Find the single biggest lever
top_change = importance_df.iloc[-1]  # last = highest delta

print('=== HEADLINE FINDING ===')
print(f"Baseline composite score:    {baseline['composite']:.3f}")
print(f"Best experiment ({best['experiment_id']}): {best['composite']:.3f}")
print(f"Total improvement:           +{total_gain:.3f} ({100*total_gain:.1f} pts)")
print()
print(f"Biggest single lever: '{top_change['change']}' → +{top_change['delta_composite']:.3f}")
print()
print(f"Best correctness:  {df['correctness'].max():.3f} ({df.loc[df['correctness'].idxmax(), 'experiment_id']})")
print(f"Best faithfulness: {df['faithfulness'].max():.3f} ({df.loc[df['faithfulness'].idxmax(), 'experiment_id']})")
print(f"Cheapest Pareto:   ${df['total_cost_usd'].min():.4f} ({df.loc[df['total_cost_usd'].idxmin(), 'experiment_id']})")

In [ ]:
save_output(df, 'experiment_summary.json')
save_output(importance_df, 'variable_importance.json')
print('Saved.')